# Banco Central / crédito rural
Acesso público à **MDCR / SICOR v2 (Olinda)** para custeio agrícola de soja, milho e cana-de-açúcar.

| Recurso | Uso no projeto |
|---|---|
| `CusteioMunicipioProduto` | Valor por município e cultura: perguntas 2-7 e extra B, após integrar a PAM (Pesquisa agrícola municipal). |
| `CusteioRegiaoUFProduto` | Valor e quantidade de contratos por UF/região e cultura: pergunta 1 e visão nacional da 2. |

**Limitação:** o recurso municipal por produto não possui `QtdCusteio`.


## SETUP - Essa seção do código é apenas o backbone para podermos construir as visualizações mais facilmente.

In [1]:
"""Configure as dependências e o acesso à API SICOR v2 do Banco Central.

Notes
-----
Define os recursos municipal e estadual, o diretório de cache, os
limites de consulta e o período de exemplo. Cria uma sessão HTTP com
timeout de conexão e leitura e retentativas para requisições GET.
Esta célula não realiza consultas à API.
"""

import hashlib
import json
import unicodedata # normalizar texto que vai/volta de/para api
import xml.etree.ElementTree as ET # nao me pergunte o que isso faz
from datetime import datetime, timezone
from functools import lru_cache
from pathlib import Path
from urllib.parse import quote, urlencode

import pandas as pd
import requests
from IPython.display import display # biblioteca legal para visualizações no notebook
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

import time

BASE = "https://olinda.bcb.gov.br/olinda/servico/SICOR/versao/v2/odata/"
RECURSO_MUNICIPAL = "CusteioMunicipioProduto"
RECURSO_UF = "CusteioRegiaoUFProduto"
RECURSOS = (RECURSO_MUNICIPAL, RECURSO_UF)
CACHE = Path("dados/bcb/cache")
TIMEOUT = (10, 90)  # conexão, leitura (segundos)
LIMITE_LOTE = 100_000
ANO_EXEMPLO, MES_EXEMPLO = 2025, 1  # teste; não define a década
ATIVIDADE_AGRICOLA = "1"

sessao = requests.Session()
sessao.headers.update({"User-Agent": "AED-Credito-Rural/1.0"})
retry = Retry(
    total=5, backoff_factor=2, raise_on_status=False,
    status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods=frozenset({"GET"}), respect_retry_after_header=True,
)
sessao.mount("https://olinda.bcb.gov.br/", HTTPAdapter(max_retries=retry))


#### SETUP - Cliente HTTP e cache auditável


In [2]:
"""Defina utilitários de consulta OData e persistência de respostas.

Notes
-----
Depende da configuração da sessão HTTP e das constantes de acesso.
As funções escapam literais, tratam falhas HTTP e mantêm um cache
auditável por URL, com detecção conservadora de possível truncamento.
"""

def literal_odata(valor):
    """Converta um valor em literal textual OData.

    Parameters
    ----------
    valor : object
        Valor convertido em texto e delimitado por aspas simples.

    Returns
    -------
    str
        Literal com aspas simples internas duplicadas.
    """
    return "'" + str(valor).replace("'", "''") + "'"


class FalhaTemporariaBCB(RuntimeError):
    """Falha recuperavel apos as retentativas HTTP (rede, 429 ou 5xx)."""


def obter_resposta(url):
    """Execute uma requisição GET com a sessão HTTP configurada.

    Parameters
    ----------
    url : str
        Endereço completo a consultar.

    Returns
    -------
    requests.Response
        Resposta aprovada pela verificação de status HTTP.

    Raises
    ------
    RuntimeError
        Se ocorrer falha de conexão, timeout ou erro de status HTTP.
    """
    try:
        resposta = sessao.get(url, timeout=TIMEOUT)
        resposta.raise_for_status()
    except requests.RequestException as exc:
        status = exc.response.status_code if exc.response is not None else None
        temporaria = (
            status in (429, 500, 502, 503, 504)
            or isinstance(exc, (requests.Timeout, requests.ConnectionError))
            and not isinstance(exc, requests.exceptions.SSLError)
        )
        erro = FalhaTemporariaBCB if temporaria else RuntimeError
        raise erro(f"Falha ao consultar BCB: {url}\n{exc}") from exc
    return resposta


@lru_cache(maxsize=16)
def _ler_cache_json(arquivo, modificado_em_ns, tamanho):
    """Reutilize o JSON em memória enquanto a versão do arquivo for a mesma.

    O caminho absoluto, a data de modificação e o tamanho identificam
    a versão. O limite evita manter todo o histórico em memória.
    """
    return json.loads(arquivo.read_text(encoding="utf-8"))


class ConsultaTruncadaError(RuntimeError):
    """A resposta pode estar incompleta e exige um recorte menor."""


def _endereco_consulta(recurso, filtro, limite):
    """Monte a URL e o caminho de cache usados por todas as consultas."""
    params = {"$format": "json", "$top": limite}
    if filtro:
        params["$filter"] = filtro
    url = BASE + recurso + "?" + urlencode(params, quote_via=quote)
    return url, CACHE / (hashlib.sha256(url.encode()).hexdigest() + ".json")


def _salvar_cache(arquivo, registro):
    """Grave o registro completo antes de substituir o arquivo de cache."""
    CACHE.mkdir(parents=True, exist_ok=True)
    temporario = arquivo.with_suffix(".tmp")
    temporario.write_text(json.dumps(registro, ensure_ascii=False), encoding="utf-8")
    temporario.replace(arquivo)


def consultar(recurso, filtro=None, *, limite=5, completo=False, atualizar=False):
    """Consulte um recurso OData com cache local e metadados de auditoria.

    Parameters
    ----------
    recurso : str
        Nome de um dos recursos definidos em ``RECURSOS``.
    filtro : str or None, optional
        Expressão enviada em ``$filter``. O padrão não aplica filtro.
    limite : int, optional
        Máximo de registros solicitado, de 1 a ``LIMITE_LOTE``. Padrão: 5.
    completo : bool, optional
        Se True, rejeita respostas com indício de truncamento. Padrão: False.
    atualizar : bool, optional
        Se True, consulta a API e substitui o cache existente. Padrão: False.

    Returns
    -------
    pandas.DataFrame
        Registros retornados, possivelmente vazios. Os atributos ``attrs``
        incluem URL, instante UTC da obtenção, opção de consulta completa
        e caminho do arquivo de cache.

    Raises
    ------
    ValueError
        Se o recurso, limite, JSON ou lista de registros for inválido,
        ou se a URL armazenada no cache não corresponder à consulta.
    RuntimeError
        Se a requisição falhar ou houver possível truncamento com
        ``completo=True``.
    OSError
        Se ocorrer falha ao ler ou gravar arquivos de cache.

    Notes
    -----
    O cache usa o hash SHA-256 da URL e não expira automaticamente.
    Os últimos 16 JSONs lidos são reutilizados em memória enquanto o
    arquivo não mudar. ``atualizar=True`` limpa esse reaproveitamento.
    Não há paginação: atingir o limite ou receber ``@odata.nextLink``
    é considerado indício de truncamento quando ``completo=True``.
    """
    if recurso not in RECURSOS:
        raise ValueError(f"Recurso não previsto: {recurso}")
    if not isinstance(limite, int) or not 1 <= limite <= LIMITE_LOTE:
        raise ValueError(f"limite deve estar entre 1 e {LIMITE_LOTE}")
    url, arquivo = _endereco_consulta(recurso, filtro, limite)
    if arquivo.exists() and not atualizar:
        estado = arquivo.stat()
        registro = _ler_cache_json(arquivo.resolve(), estado.st_mtime_ns, estado.st_size)
        if registro["url"] != url:
            raise ValueError("Cache incompatível com a consulta.")
    else:
        if atualizar:
            _ler_cache_json.cache_clear()
        resposta = obter_resposta(url)
        try:
            dados = resposta.json()
        except ValueError as exc:
            raise ValueError("O BCB não retornou JSON válido.") from exc
        if not isinstance(dados, dict) or not isinstance(dados.get("value"), list):
            raise ValueError("Resposta OData sem a lista 'value'.")
        registro = {
            "url": url, "obtido_em_utc": datetime.now(timezone.utc).isoformat(),
            "dados": dados,
        }
        _salvar_cache(arquivo, registro)
    dados = registro["dados"]
    if not isinstance(dados.get("value"), list):
        raise ValueError("Cache inválido: lista 'value' ausente.")
    if completo and (len(dados["value"]) >= limite or dados.get("@odata.nextLink")):
        raise ConsultaTruncadaError("Possível truncamento. Reduza o recorte; não use $skip.")
    df = pd.DataFrame(dados["value"])
    df.attrs.update({
        "url": url, "obtido_em_utc": registro["obtido_em_utc"],
        "consulta_completa": completo, "arquivo_cache": str(arquivo),
    })
    for campo in ("url_origem", "urls_origem"):
        if campo in registro:
            df.attrs[campo] = registro[campo]
    return df


#### SETUP - Conferir o esquema oficial
Identificamos o tipo associado a cada recurso sem pressupor seus nomes internos. A chave municipal para integração é `codIbge`, preservada como texto. `codCadMu` e `cdEstado` são códigos do BCB, não códigos IBGE.


In [3]:
"""Consulte e confira o esquema dos recursos de custeio.

Notes
-----
Depende do cliente HTTP definido anteriormente. Consulta ``$metadata``,
salva o XML no cache, exibe os campos e verifica as colunas essenciais
com assertivas. Informa se o recurso municipal contém ``QtdCusteio``.
"""

def ler_esquema():
    """Leia os campos e tipos dos recursos no esquema OData oficial.

    Returns
    -------
    pandas.DataFrame
        Tabela com as colunas ``recurso``, ``campo`` e ``tipo``, contendo
        as propriedades dos recursos definidos em ``RECURSOS``.

    Raises
    ------
    RuntimeError
        Se a consulta HTTP ao esquema falhar.
    xml.etree.ElementTree.ParseError
        Se a resposta não contiver XML válido.
    ValueError
        Se algum recurso esperado não tiver propriedades no esquema.
    KeyError
        Se faltar um atributo XML obrigatório ou um tipo referenciado.
    OSError
        Se não for possível gravar o esquema no cache.

    Notes
    -----
    Consulta a API em cada chamada e salva o XML em ``CACHE/metadata.xml``.
    """
    xml = obter_resposta(BASE + "$metadata").content
    raiz = ET.fromstring(xml)
    ns = {"edm": "http://docs.oasis-open.org/odata/ns/edm"}
    tipos = {item.attrib["Name"]: item for item in raiz.findall(".//edm:EntityType", ns)}
    linhas = []
    for entidade in raiz.findall(".//edm:EntitySet", ns):
        recurso = entidade.attrib["Name"]
        if recurso in RECURSOS:
            tipo = tipos[entidade.attrib["EntityType"].rsplit(".", 1)[-1]]
            for campo in tipo.findall("edm:Property", ns):
                linhas.append({"recurso": recurso, "campo": campo.attrib["Name"],
                               "tipo": campo.attrib["Type"]})
    esquema = pd.DataFrame(linhas)
    if esquema.empty or set(esquema["recurso"]) != set(RECURSOS):
        raise ValueError("Recursos esperados não encontrados no esquema.")
    CACHE.mkdir(parents=True, exist_ok=True)
    (CACHE / "metadata.xml").write_bytes(xml)
    return esquema


esquema = ler_esquema()
display(esquema)
campos_municipais = set(esquema.loc[esquema["recurso"].eq(RECURSO_MUNICIPAL), "campo"])
campos_uf = set(esquema.loc[esquema["recurso"].eq(RECURSO_UF), "campo"])
assert {"codIbge", "nomeProduto", "VlCusteio", "AnoEmissao"} <= campos_municipais
assert {"nomeUF", "nomeRegiao", "QtdCusteio", "VlCusteio"} <= campos_uf
print("Quantidade de contratos no recurso municipal:", "QtdCusteio" in campos_municipais)


,recurso,campo,tipo
0,CusteioMunicipioProduto,Municipio,Edm.String
1,CusteioMunicipioProduto,nomeProduto,Edm.String
2,CusteioMunicipioProduto,MesEmissao,Edm.String
3,CusteioMunicipioProduto,AnoEmissao,Edm.String
4,CusteioMunicipioProduto,cdPrograma,Edm.String
5,CusteioMunicipioProduto,cdSubPrograma,Edm.String
6,CusteioMunicipioProduto,cdFonteRecurso,Edm.String
7,CusteioMunicipioProduto,cdTipoSeguro,Edm.String
8,CusteioMunicipioProduto,cdEstado,Edm.String
9,CusteioMunicipioProduto,VlCusteio,Edm.Decimal


Quantidade de contratos no recurso municipal: False


#### SETUP - Identificar culturas em pequenas amostras

Janeiro/2025 é apenas um teste.


In [4]:
"""Identifique os nomes de produtos das três culturas em amostras.

Notes
-----
Depende das funções de consulta e das constantes do período de exemplo.
Define a normalização de nomes e os filtros mensais, consulta amostras,
exibe o catálogo e cria ``produtos`` para as coletas posteriores.
A execução exige que soja, milho e cana-de-açúcar sejam identificados.
"""

def normalizar_nome(valor):
    """Padronize um nome de produto para comparação com as culturas.

    Parameters
    ----------
    valor : object
        Nome convertido em texto antes da normalização.

    Returns
    -------
    str
        Texto em maiúsculas, sem acentos ou aspas duplas nas extremidades,
        com hífens substituídos por espaços e espaçamento consolidado.
    """
    texto = unicodedata.normalize("NFKD", str(valor).strip().strip('"').upper())
    return " ".join("".join(c for c in texto if not unicodedata.combining(c))
                    .replace("-", " ").split())


CULTURAS = {"SOJA": "soja", "MILHO": "milho", "CANA DE ACUCAR": "cana_de_acucar"}


def filtro_periodo(ano, mes):
    """Monte o filtro OData de período para a atividade agrícola.

    Parameters
    ----------
    ano : int
        Ano de emissão entre 1900 e 2100, inclusive.
    mes : int
        Mês de emissão entre 1 e 12, inclusive.

    Returns
    -------
    str
        Expressão para ``AnoEmissao``, ``MesEmissao`` e ``Atividade``,
        com mês de dois dígitos e valores como literais textuais.

    Raises
    ------
    ValueError
        Se ano ou mês não for inteiro ou estiver fora do intervalo aceito.
    """
    if not isinstance(ano, int) or not 1900 <= ano <= 2100:
        raise ValueError("Ano inválido.")
    if not isinstance(mes, int) or not 1 <= mes <= 12:
        raise ValueError("Mês inválido.")
    return (f"AnoEmissao eq {literal_odata(ano)} and "
            f"MesEmissao eq {literal_odata(f'{mes:02d}')} and "
            f"Atividade eq {literal_odata(ATIVIDADE_AGRICOLA)}")


def descobrir_produtos(ano, mes):
    """Amostre produtos municipais e associe seus nomes às culturas.

    Parameters
    ----------
    ano : int
        Ano de emissão entre 1900 e 2100, inclusive.
    mes : int
        Mês de emissão entre 1 e 12, inclusive.

    Returns
    -------
    pandas.DataFrame
        Catálogo sem duplicatas com ``nomeProduto``, ``cdProduto``,
        ``Atividade`` e ``cultura``. Nomes não mapeados têm cultura ausente.

    Raises
    ------
    ValueError
        Se o período for inválido, todas as amostras forem vazias ou
        a consulta retornar dados inválidos.
    RuntimeError
        Se uma consulta HTTP falhar.

    Notes
    -----
    Consulta até dez registros por termo: SOJA, MILHO e CANA.
    O resultado é amostral e não garante cobertura de todas as culturas.
    """
    amostras = []
    for termo in ("SOJA", "MILHO", "CANA"):
        filtro = filtro_periodo(ano, mes) + f" and contains(nomeProduto,{literal_odata(termo)})"
        amostra = consultar(RECURSO_MUNICIPAL, filtro, limite=10)
        if not amostra.empty:
            amostras.append(amostra[["nomeProduto", "cdProduto", "Atividade"]])
    if not amostras:
        raise ValueError("Amostras vazias. Confira o período ou tente novamente.")
    catalogo = pd.concat(amostras, ignore_index=True).drop_duplicates()
    catalogo["cultura"] = catalogo["nomeProduto"].map(normalizar_nome).map(CULTURAS)
    return catalogo.reset_index(drop=True) # drop=True -> para tirar a coluna de indice anterior


catalogo_amostra = descobrir_produtos(ANO_EXEMPLO, MES_EXEMPLO)
display(catalogo_amostra)
produtos = (catalogo_amostra.dropna(subset=["cultura"])[["nomeProduto", "cultura"]]
            .drop_duplicates().reset_index(drop=True))
if set(produtos["cultura"]) != set(CULTURAS.values()):
    raise ValueError("A amostra não identificou as três culturas. Revise as categorias.")


,nomeProduto,cdProduto,Atividade,cultura
0,"""SOJA""",6720,1,soja
1,"""MILHO""",5080,1,milho
2,"""CANA-DE-AÇUCAR""",1840,1,cana_de_acucar


#### SETUP - Coleta por lote e preparação

In [ ]:
"""Defina a coleta mensal, o tratamento e a agregação anual do crédito.

Notes
-----
Depende do cliente OData, do mapeamento de culturas e dos filtros.
As funções validam lotes, coletam períodos com auditoria e agregam
valores por localidade, cultura e ano, preservando totais desconhecidos.
Esta célula apenas define funções; a coleta ocorre ao chamá-las.
"""

def preparar_lote(bruto, recurso):
    """Valide e padronize um lote de custeio municipal ou estadual.

    Parameters
    ----------
    bruto : pandas.DataFrame
        Registros originais com produto, período, atividade, valor e
        campos geográficos exigidos pelo recurso.
    recurso : str
        Nome de um dos recursos definidos em ``RECURSOS``.

    Returns
    -------
    pandas.DataFrame
        Cópia com cultura, ano, mês, crédito nominal em reais e quantidade
        de contratos padronizados. Inclui códigos IBGE para municípios
        ou nomes de UF e região para o recurso estadual. Se a entrada
        estiver vazia, retorna um DataFrame vazio sem colunas.

    Raises
    ------
    ValueError
        Se o recurso, os campos obrigatórios, a cultura, a atividade,
        o período, a localização ou os valores numéricos forem inválidos.
    TypeError
        Se uma conversão para inteiro anulável não puder ser realizada.

    Notes
    -----
    Mantém as colunas originais. Valores monetários e quantidades ausentes
    são preservados; valores negativos são rejeitados. No recurso
    municipal, a quantidade de contratos permanece ausente e o código
    IBGE deve conter sete dígitos, sem preenchimento automático de zeros.
    """
    if recurso not in RECURSOS:
        raise ValueError("Recurso fora do escopo.")
    if bruto.empty:
        return pd.DataFrame()
    obrigatorios = {"nomeProduto", "AnoEmissao", "MesEmissao", "Atividade", "VlCusteio"}
    obrigatorios |= ({"codIbge"} if recurso == RECURSO_MUNICIPAL
                     else {"nomeUF", "nomeRegiao", "QtdCusteio"})
    faltantes = obrigatorios - set(bruto.columns)
    if faltantes:
        raise ValueError(f"Campos ausentes: {sorted(faltantes)}")
    df = bruto.copy()
    # Cada nome se repete em muitas linhas; normalize apenas os distintos.
    culturas_por_nome = {nome: CULTURAS.get(normalizar_nome(nome))
                         for nome in df["nomeProduto"].unique()}
    df["cultura"] = df["nomeProduto"].map(culturas_por_nome)
    # jesus isso está muito feio...
    if df["cultura"].isna().any() or not df["Atividade"].eq("1").all():
        raise ValueError("Produto não mapeado ou atividade não agrícola.")
    for origem, destino in (("AnoEmissao", "ano"), ("MesEmissao", "mes")):
        df[destino] = pd.to_numeric(df[origem], errors="raise").astype("Int64")
        if df[destino].isna().any():
            raise ValueError(f"{origem} ausente.")
    if not df["mes"].between(1, 12).all():
        raise ValueError("Mês fora do intervalo 1–12.")
    df["credito_nominal_brl"] = pd.to_numeric(df["VlCusteio"], errors="raise").astype("Float64")
    if df["credito_nominal_brl"].dropna().lt(0).any():
        raise ValueError("Valor negativo: investigar antes de agregar.")
    if recurso == RECURSO_MUNICIPAL:
        df["codigo_municipio"] = df["codIbge"].astype("string").str.strip()
        if not df["codigo_municipio"].str.fullmatch(r"\d{7}", na=False).all():
            print("Código IBGE ausente ou inválido; cortando a linha em vez de interromper.")
            df = df[df["codigo_municipio"].str.fullmatch(r"\d{7}", na=False)]
            # raise ValueError("Código IBGE ausente ou inválido; não preencher com zeros.")
        df["codigo_uf_ibge"] = df["codigo_municipio"].str[:2]
        df["quantidade_contratos"] = pd.Series(pd.NA, index=df.index, dtype="Int64")
    else:
        df["uf"] = df["nomeUF"].astype("string").str.strip()
        df["regiao"] = df["nomeRegiao"].astype("string").str.strip()
        if df[["uf", "regiao"]].isna().any().any() or df["uf"].eq("").any() or df["regiao"].eq("").any():
            raise ValueError("UF ou região ausente.")
        df["quantidade_contratos"] = pd.to_numeric(df["QtdCusteio"], errors="raise").astype("Int64")
        if df["quantidade_contratos"].dropna().lt(0).any():
            raise ValueError("Quantidade negativa.")
    return df

def coletar_lote(recurso, ano, mes, nome_produto, *, atualizar=False):
    """Colete e valide um lote mensal de um produto, com auditoria.

    Parameters
    ----------
    recurso : str
        Nome de um dos recursos definidos em ``RECURSOS``.
    ano : int
        Ano de emissão entre 1900 e 2100, inclusive.
    mes : int
        Mês de emissão entre 1 e 12, inclusive.
    nome_produto : str
        Nome exato retornado pela API, incluindo eventuais aspas, cuja
        forma normalizada corresponda a uma cultura de ``CULTURAS``.
    atualizar : bool, optional
        Se True, substitui o cache da consulta. Padrão: False.

    Returns
    -------
    tratado : pandas.DataFrame
        Lote padronizado, possivelmente vazio.
    auditoria : dict
        Recurso, período, produto, número de linhas brutas e atributos
        da consulta, incluindo URL, instante UTC e arquivo de cache.

    Raises
    ------
    ValueError
        Se os argumentos ou dados forem inválidos, ou se os registros
        retornados não respeitarem o período e o produto solicitados.
    RuntimeError
        Se a consulta HTTP falhar ou houver indício de truncamento.

    Notes
    -----
    Solicita até ``LIMITE_LOTE`` registros com ``completo=True``.
    Reaproveita o cache local e normaliza cada nome de produto uma
    única vez por lote, mantendo as validações e a auditoria.
    """
    if recurso not in RECURSOS or normalizar_nome(nome_produto) not in CULTURAS:
        raise ValueError("Recurso ou produto fora do escopo.")
    filtro = filtro_periodo(ano, mes) + f" and nomeProduto eq {literal_odata(nome_produto)}"
    try:
        bruto = consultar(recurso, filtro, limite=LIMITE_LOTE, completo=True, atualizar=atualizar)
    except (FalhaTemporariaBCB, ConsultaTruncadaError):
        if recurso != RECURSO_MUNICIPAL:
            raise
        print(f"BCB: dividindo {ano}-{mes:02d} / {nome_produto} por estado.")
        bruto = _consultar_por_estado(recurso, filtro, atualizar=atualizar)
    tratado = preparar_lote(bruto, recurso)
    if not tratado.empty:
        if not (tratado["ano"].eq(ano) & tratado["mes"].eq(mes)
                & tratado["nomeProduto"].eq(nome_produto)).all():
            raise ValueError("Resposta não respeita o filtro solicitado.")
    auditoria = {"recurso": recurso, "ano": ano, "mes": mes, "nome_produto": nome_produto,
                 "linhas": len(bruto), **bruto.attrs}
    return tratado, auditoria


def _consultar_por_estado(recurso, filtro, *, atualizar=False):
    """Recupere um lote municipal em tres recortes disjuntos e exaustivos.

    cdEstado e textual: as faixas usam a mesma comparacao da API e
    incluem nulos, sem depender de uma lista fixa de estados. Cada parte
    tem seu proprio cache. O cache do lote so e substituido apos todas
    as partes estarem completas; uma falha nunca vira um lote vazio.
    """
    faixas = (
        "(cdEstado lt '15' or cdEstado eq null)",
        "(cdEstado ge '15' and cdEstado lt '24')",
        "cdEstado ge '24'",
    )
    linhas, urls = [], []
    for faixa in faixas:
        parte = consultar(recurso, f"({filtro}) and ({faixa})",
                          limite=LIMITE_LOTE, completo=True, atualizar=atualizar)
        arquivo = Path(parte.attrs["arquivo_cache"])
        estado = arquivo.stat()
        registro = _ler_cache_json(arquivo.resolve(), estado.st_mtime_ns, estado.st_size)
        linhas.extend(registro["dados"]["value"])
        urls.append(registro["url"])
    # Nao deduplicar: linhas iguais podem representar contratos distintos.
    # Mantenha a verificacao conservadora do limite do lote consolidado.
    if len(linhas) >= LIMITE_LOTE:
        raise ConsultaTruncadaError("Lote consolidado atingiu LIMITE_LOTE; reduza o recorte.")
    url, arquivo = _endereco_consulta(recurso, filtro, LIMITE_LOTE)
    _salvar_cache(arquivo, {
        "url": url, "urls_origem": urls,
        "obtido_em_utc": datetime.now(timezone.utc).isoformat(),
        "dados": {"value": linhas},
    })
    return consultar(recurso, filtro, limite=LIMITE_LOTE, completo=True)


def _preparar_cache_mensal(recurso, ano, mes, nomes, *, atualizar=False):
    """Consulte juntos os produtos pendentes e grave seus caches individuais.

    Retorna True se o grupo foi concluído. Falhas temporarias e respostas possivelmente
    truncadas ficam para as consultas individuais de coletar_periodo.
    """
    periodo = filtro_periodo(ano, mes)
    pendentes = {}
    for nome in nomes:
        filtro = periodo + f" and nomeProduto eq {literal_odata(nome)}"
        url, arquivo = _endereco_consulta(recurso, filtro, LIMITE_LOTE)
        if atualizar or not arquivo.exists():
            pendentes[nome] = (url, arquivo)
    if len(pendentes) < 2:
        return False
    produtos = " or ".join(f"nomeProduto eq {literal_odata(nome)}"
                           for nome in sorted(pendentes))
    try:
        bruto = consultar(recurso, periodo + f" and ({produtos})",
                          limite=LIMITE_LOTE, completo=True, atualizar=atualizar)
    except (ConsultaTruncadaError, FalhaTemporariaBCB):
        return False
    if not bruto.empty:
        campos = {"nomeProduto", "AnoEmissao", "MesEmissao"}
        if not campos.issubset(bruto.columns) or not (
            bruto["nomeProduto"].isin(pendentes)
            & pd.to_numeric(bruto["AnoEmissao"], errors="raise").eq(ano)
            & pd.to_numeric(bruto["MesEmissao"], errors="raise").eq(mes)
        ).all():
            raise ValueError("Resposta agrupada não respeita o filtro solicitado.")
    # Separe o JSON original para preservar tipos e valores da resposta.
    arquivo_grupo = Path(bruto.attrs["arquivo_cache"])
    estado = arquivo_grupo.stat()
    registro = _ler_cache_json(arquivo_grupo.resolve(), estado.st_mtime_ns, estado.st_size)
    por_nome = {nome: [] for nome in pendentes}
    for linha in registro["dados"]["value"]:
        por_nome[linha["nomeProduto"]].append(linha)
    for nome, (url, arquivo) in pendentes.items():
        _salvar_cache(arquivo, {
            "url": url, "url_origem": registro["url"],
            "obtido_em_utc": registro["obtido_em_utc"],
            "dados": {"value": por_nome[nome]},
        })
    return True


def coletar_periodo(recurso, anos, produtos, *, atualizar=False):
    """Colete todos os meses dos anos e produtos informados.

    Parameters
    ----------
    recurso : str
        Nome de um dos recursos definidos em ``RECURSOS``.
    anos : iterable of int
        Anos distintos entre 1900 e 2100. A coleção não pode ser vazia.
    produtos : pandas.DataFrame
        Catálogo com a coluna ``nomeProduto`` contendo nomes exatos
        validados para as culturas de interesse. Duplicatas são removidas.
    atualizar : bool, optional
        Se True, atualiza o cache de cada lote. Padrão: False.

    Returns
    -------
    dados : pandas.DataFrame
        Concatenação dos lotes tratados não vazios, com índice reiniciado.
        Retorna um DataFrame vazio sem colunas se todos os lotes forem vazios.
    auditoria : pandas.DataFrame
        Um registro por consulta, incluindo respostas vazias.

    Raises
    ------
    ValueError
        Se os anos forem vazios ou repetidos, não houver produtos, ou
        um lote apresentar argumentos ou dados inválidos.
    KeyError
        Se o catálogo não contiver a coluna ``nomeProduto``.
    RuntimeError
        Se uma consulta falhar ou houver indício de truncamento.

    Notes
    -----
    Percorre os anos em ordem crescente e imprime o progresso anual.
    Agrupa por mês os produtos ausentes do cache em uma chamada HTTP.
    Se o grupo falhar temporariamente ou puder estar truncado, consulta
    os produtos separadamente. Lotes municipais que ainda falharem sao
    divididos por faixas de estado, preservando as URLs na auditoria.
    Preserva caches individuais, inclusive vazios, com a URL de origem
    agrupada em ``url_origem`` na auditoria.
    Uma falha interrompe a coleta; caches já gravados permanecem em disco.
    """
    anos = list(anos)
    if not anos or len(anos) != len(set(anos)):
        raise ValueError("Informe anos distintos.")
    nomes = produtos["nomeProduto"].drop_duplicates().tolist()
    if not nomes:
        raise ValueError("Informe produtos validados.")
    if recurso not in RECURSOS or any(normalizar_nome(nome) not in CULTURAS for nome in nomes):
        raise ValueError("Recurso ou produto fora do escopo.")
    for ano in anos:
        filtro_periodo(ano, 1)
    lotes, auditoria = [], []
    for ano in sorted(anos):
        print("Coletando dados para o ano:", ano)
        for mes in range(1, 13):
            agrupado = _preparar_cache_mensal(recurso, ano, mes, nomes, atualizar=atualizar)
            for nome in nomes:
                lote, registro = coletar_lote(recurso, ano, mes, nome,
                                              atualizar=atualizar and not agrupado)
                auditoria.append(registro)  # inclui respostas vazias
                if not lote.empty:
                    lotes.append(lote)
        print(f"{recurso}: {ano} consultado.")
    dados = pd.concat(lotes, ignore_index=True) if lotes else pd.DataFrame()
    return dados, pd.DataFrame(auditoria)


def agregar_anual(dados, recurso):
    """Agregue crédito e contratos por localidade, cultura e ano.

    Parameters
    ----------
    dados : pandas.DataFrame
        Lotes tratados com chaves geográficas, cultura, ano, crédito
        nominal em reais e quantidade de contratos.
    recurso : str
        Recurso municipal ou estadual que determina as chaves geográficas.

    Returns
    -------
    pandas.DataFrame
        Totais ordenados pelas chaves geográficas, cultura e ano, com
        índice reiniciado. Cada total fica ausente se alguma parcela
        da respectiva medida estiver ausente no grupo.

    Raises
    ------
    ValueError
        Se o recurso estiver fora do escopo ou os dados estiverem vazios.
    KeyError
        Se faltar uma coluna necessária ao agrupamento ou à agregação.

    Notes
    -----
    Inclui grupos com chaves ausentes. Não verifica a presença dos doze
    meses nem detecta registros faltantes ou duplicados na entrada.
    """
    if recurso not in RECURSOS:
        raise ValueError("Recurso fora do escopo.")
    if dados.empty:
        raise ValueError("Sem dados para agregar.")
    chaves = (["codigo_municipio", "codigo_uf_ibge"] if recurso == RECURSO_MUNICIPAL
              else ["uf", "regiao"]) + ["cultura", "ano"]
    def somar_completos(serie):
        """Some uma medida somente quando todas as parcelas estiverem presentes.

        Parameters
        ----------
        serie : pandas.Series
            Valores numéricos de uma medida dentro de um grupo.

        Returns
        -------
        scalar
            Soma dos valores ou marcador de ausência compatível com o
            dtype da série quando houver alguma parcela ausente.
        """
        # Uma parcela ausente torna o total desconhecido, evitando subestimação.
        return serie.sum(min_count=len(serie))
    return (dados.groupby(chaves, as_index=False, dropna=False)[
                ["credito_nominal_brl", "quantidade_contratos"]]
            .agg(somar_completos).sort_values(chaves).reset_index(drop=True))


## VISUALIZAÇÕES
### Nesta seção serão construídas as visualizações que responderam as perguntas feitas em nosso relatório

In [7]:
import altair as alt
import numpy as np
